# Unsupervised Anomaly Detection (Clustering) Graph Neural Net

### References:  
Converting tabular data to graph data for GNN: https://youtu.be/AQU3akndun4?si=g8V4zo36SQNdGLW-   
Fake News detection using GNN: https://youtu.be/QAIVFr24FrA?si=cw2sDav_jZJMuaBD   
Self-/Unsupervised GNN training: https://youtu.be/3XTuhchTWd8?si=Jxi8Xn9hPOD6EQQF 

# Systematisation:  
Heterogeneous graph for Spain dataset  
<b>Nodes:</b> customer, merchant   
<b>Node features:</b> age, gender  
<b>Edges:</b> each transaction  
<b>Edge features:</b> category, amount  
<b>Target label:</b> fraud

### 1. Data Cleaning

In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.data import HeteroData
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
df = pd.read_csv('../data/raw/banksim.csv')

In [ ]:
# remove single quotes from string columns
for col in ['customer', 'age', 'gender', 'merchant', 'category']:
    df[col] = df[col].astype(str).str.replace("'", "")

In [ ]:
# handle 'U' in age - replce w mode
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['age'] = df['age'].fillna(df['age'].mode()[0])

### 2. Encoding + Feature Engineering

In [ ]:
# encode customer and merchant IDs to integers
customer_encoder = LabelEncoder()
merchant_encoder = LabelEncoder()

In [ ]:
# fit encoding to full dataset
df['customer_id'] = customer_encoder.fit_transform(df['customer'])
df['merchant_id'] = merchant_encoder.fit_transform(df['merchant'])

In [ ]:
# one-hot encode gender ('M', 'F', 'U', 'E')
gender_dummies = pd.get_dummies(df['gender'], prefix='gen')
df = pd.concat([df, gender_dummies], axis=1)

In [ ]:
# one-hot encode category
category_dummies = pd.get_dummies(df['category'], prefix='cat')
df = pd.concat([df, category_dummies], axis=1)

In [ ]:
# count transactions to add popularity as a feature for merchants
merch_counts = df.groupby('merchant_id').size().reset_index(name='popularity')
df = df.merge(merch_counts, on='merchant_id')

In [ ]:
df = df.drop(columns=['customer', 'merchant', 'zipcodeori', 'zipmerchant', 'gender', 'category'])
df

In [ ]:
data = HeteroData()

In [ ]:
# customer features (age, gender (f, m, u, e))
cust_feature_cols = ['age'] + [col for col in df.columns if col.startswith('gen_')]
cust_features_df = df.groupby('customer_id')[cust_feature_cols].first().sort_index()
data['customer'].x = torch.tensor(cust_features_df.values.astype(float), dtype=torch.float)

In [ ]:
# scale merchant popularity
merch_scaler = StandardScaler()

In [ ]:
# use only non-fraudulent transactions for training
train_df = df[df['fraud'] == 0].copy()

In [ ]:
# merchant features (adding popularity)
merch_features = train_df.groupby('merchant_id')['popularity'].first().sort_index().values.reshape(-1, 1)
merch_features_scaled = merch_scaler.fit_transform(merch_features)
data['merchant'].x = torch.tensor(merch_features_scaled.astype(float), dtype=torch.float)

In [ ]:
# edges: customer -> merchant (using only non-fraud transactions)
edge_index = torch.stack([
    torch.tensor(train_df['customer_id'].values, dtype=torch.long),
    torch.tensor(train_df['merchant_id'].values, dtype=torch.long)
], dim=0)

In [ ]:
data['customer', 'buys_from', 'merchant'].edge_index = edge_index

In [ ]:
# edge attributes (amount, category)
scaler_amount = StandardScaler()
amounts = scaler_amount.fit_transform(train_df[['amount']])

category_cols = [col for col in df.columns if col.startswith('cat_')]

edge_features = np.hstack([amounts, train_df[category_cols].values])

In [ ]:
data['customer', 'buys_from', 'merchant'].edge_attr = torch.tensor(edge_features.astype(float), dtype=torch.float)

In [ ]:
print(f"Customer mean: {data['customer'].x.mean()}\nMerchant mean: {data['merchant'].x.mean()}")

In [ ]:
print(f"Edge feature shape: {data['customer', 'buys_from', 'merchant'].edge_attr.shape}")

### Unsupervised GNN Model (GraphSAGE Encoder)

In [ ]:
from torch_geometric.nn import SAGEConv, to_hetero
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class AnomalyEncoder(nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

In [ ]:
import torch_geometric.transforms as T

# reverse edge types
data = T.ToUndirected()(data)
print(data.metadata()) 

In [ ]:
# init model and convert to heterogenous
model = AnomalyEncoder(hidden_channels=16, out_channels=32)
model = to_hetero(model, data.metadata(), aggr='mean')

In [ ]:
# optimiser = adam
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

### Train the Model

In [ ]:
# unsupervised reconstructive loss
# nodes that are connected should have similar embeddings

loss_history = []
model.train()

for epoch in range(1, 101):
    optimizer.zero_grad()
    
    # generate embeddings
    z_dict = model(data.x_dict, data.edge_index_dict)

    # link reconstruction loss
    pos_src = z_dict['customer'][data['customer', 'buys_from', 'merchant'].edge_index[0]]
    pos_dst = z_dict['merchant'][data['customer', 'buys_from', 'merchant'].edge_index[1]]
    pos_score = (pos_src * pos_dst).sum(dim=-1)

    # 2. Negative Samples (Fake Edges)
    # We pick a random merchant for every real customer in the edge_index
    num_edges = data['customer', 'buys_from', 'merchant'].edge_index.size(1)
    random_indices = torch.randint(0, data['merchant'].num_nodes, (num_edges,), device=data['merchant'].x.device)

    neg_src = z_dict['customer'][data['customer', 'buys_from', 'merchant'].edge_index[0]]
    neg_dst = z_dict['merchant'][random_indices]
    neg_score = (neg_src * neg_dst).sum(dim=-1)

    # 3. Stable Binary Cross Entropy Loss
    # We use logsigmoid for better numerical stability than log(sigmoid)
    pos_loss = -torch.nn.functional.logsigmoid(pos_score).mean()
    neg_loss = -torch.nn.functional.logsigmoid(-neg_score).mean()
    loss = pos_loss + neg_loss
    
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 10 == 0:
        print(f'Epoch {epoch}: Loss = {loss.item():.4f}')

In [ ]:
# plot loss history curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, 101), loss_history, color='#2ca02c', linewidth=2)

# Aesthetics
plt.title('GNN Training Loss (Unsupervised Reconstruction)', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

# Adding an annotation for the "Elbow" (where learning stabilizes)
plt.annotate('Model Convergence', xy=(80, loss_history[-1]), xytext=(60, loss_history[0]/2),
             arrowprops=dict(facecolor='black', shrink=0.05))

plt.show()

### Clustering + Evaluation

In [ ]:
from sklearn.ensemble import IsolationForest

In [ ]:
# build edge scores for all transactions (including frauds)
full_data = HeteroData()

# nodes stay the same
full_data['customer'].x = data['customer'].x
full_data['merchant'].x = data['merchant'].x

# use edges from full dataset, not just train_df
full_edge_index = torch.stack([
    torch.tensor(df['customer_id'].values, dtype=torch.long),
    torch.tensor(df['merchant_id'].values, dtype=torch.long)
], dim=0)
full_data['customer', 'buys_from', 'merchant'].edge_index = full_edge_index

# edge attributes (amount, category) - scale using the same scaler fitted on train_df
full_amounts = scaler_amount.transform(df[['amount']])
full_edge_feat = np.hstack([full_amounts, df[category_cols].values])
full_data['customer', 'buys_from', 'merchant'].edge_attr = torch.tensor(full_edge_feat.astype(float), dtype=torch.float)

# make it undirected to match the model architecture
full_data = T.ToUndirected()(full_data)

In [ ]:
model.eval()

In [ ]:
with torch.no_grad():
    # get embeddings for all customers (incl fraud)
    all_z = model(full_data.x_dict, full_data.edge_index_dict)
    cust_embedding = all_z['customer'].numpy()

# Isolation Forest (Classifier)  
Isolation Forest is an unsupervised anomaly detection method that identifies anomalies by isolating observations in the feature space. 
It works by randomly selecting a feature and then randomly selecting a split value between the maximum and minimum values of that feature. 
This process is repeated recursively until all observations are isolated.  
  
Anomalies are more likely to be isolated earlier in the process, resulting in shorter path lengths in the isolation tree. 
The Isolation Forest algorithm uses these path lengths to assign an anomaly score to each observation, with higher scores indicating a higher likelihood of being an anomaly.  
  
By training the Isolation Forest on the embeddings of normal transactions, we can identify fraudulent transactions as those with high anomaly scores. 
We need this because the GNN only converts graph relationships into embeddings, it doesn't directly classify transactions as fraud or not.

In [ ]:
# train Isolation Forest on normal embeddings
iso_forest = IsolationForest(contamination=0.01, random_state=42)
iso_forest.fit(cust_embedding)

In [ ]:
# make the prediction: 1 is normal, -1 is anomaly
anomaly_labels = iso_forest.predict(cust_embedding)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
# map isolation forest labels from -1,1 to 1,0
inferred_fraud = np.where(anomaly_labels == -1, 1, 0)

pred_map = {i: pred for i, pred in enumerate(inferred_fraud)}
df['predicted_fraud'] = df['customer_id'].map(pred_map)

In [ ]:
print(f'Classification Report:\n {classification_report(df["fraud"], df["predicted_fraud"])}')

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Reduce dimensions to 2D
tsne = TSNE(n_components=2, random_state=42)
vis_dims = tsne.fit_transform(cust_embedding)

# Plotting
plt.figure(figsize=(10, 7))
# Plot normal cases
plt.scatter(vis_dims[inferred_fraud == 0, 0], vis_dims[inferred_fraud == 0, 1], 
            c='blue', alpha=0.5, label='Normal Cluster')
# Plot predicted anomalies
plt.scatter(vis_dims[inferred_fraud == 1, 0], vis_dims[inferred_fraud == 1, 1], 
            c='red', edgecolors='black', label='Detected Anomalies')

plt.title("GNN Embedding Space: Normal vs. Anomalous Customers")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1. Plot the Confusion Matrix
cm = confusion_matrix(df['fraud'], df['predicted_fraud'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Fraud'])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix: Unsupervised GNN + Isolation Forest")
plt.show()

# 2. Summary Statistics
print(f"Total Anomalies Detected: {sum(inferred_fraud)}")
print(f"Actual Fraud Cases in Dataset: {sum(df['fraud'])}")